In [ ]:
import kagglehub

path = kagglehub.dataset_download("ravi72munde/uber-lyft-cab-prices")
print("Path to dataset files:", path)

100%|██████████| 73.5M/73.5M [00:00<00:00, 121MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/ravi72munde/uber-lyft-cab-prices/versions/4


In [ ]:
import os

os.listdir(path)

['weather.csv', 'cab_rides.csv', 'Cab-Weather Data', 'cab-weather data']

In [ ]:
import pandas as pd
import os

# Find correct file path
for root, dirs, files in os.walk(path):
    for file in files:
        if file == "cab_rides.csv":
            full_path = os.path.join(root, file)

print(full_path)

/root/.cache/kagglehub/datasets/ravi72munde/uber-lyft-cab-prices/versions/4/cab_rides.csv


In [ ]:
cab_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 637976 entries, 0 to 693070
Data columns (total 8 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   distance          637976 non-null  float64
 1   cab_type          637976 non-null  object 
 2   time_stamp        637976 non-null  int64  
 3   destination       637976 non-null  object 
 4   source            637976 non-null  object 
 5   price             637976 non-null  float64
 6   surge_multiplier  637976 non-null  float64
 7   name              637976 non-null  object 
dtypes: float64(3), int64(1), object(4)
memory usage: 43.8+ MB


In [ ]:
# Only drop rows where price is missing
cab_data = cab_data.dropna(subset=["price"])

# Drop unnecessary columns
cab_data = cab_data.drop(columns=[
    "id", "product_id", "timestamp", "datetime"
], errors="ignore")

X = cab_data.drop("price", axis=1)
y = cab_data["price"]

X = pd.get_dummies(X, drop_first=True)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)



In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

MAE: 1.749157517962254
RMSE: 2.4935997758837103
R²: 0.9286761906289557


In [ ]:
import numpy as np

def bootstrap_ci(y_true, y_pred, n_bootstrap=1000):
    errors = []
    n = len(y_true)

    for _ in range(n_bootstrap):
        indices = np.random.choice(range(n), n, replace=True)
        error = np.mean(np.abs(y_true.iloc[indices] - y_pred[indices]))
        errors.append(error)

    return np.percentile(errors, [2.5, 97.5])

ci = bootstrap_ci(y_test.reset_index(drop=True), y_pred)
print("95% CI for MAE:", ci)

95% CI for MAE: [1.73944811 1.75821651]


In [ ]:
# IMPORT LIBRARIES
import kagglehub
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# DOWNLOAD DATASET
path = kagglehub.dataset_download("ravi72munde/uber-lyft-cab-prices")

# Locate cab_rides.csv
for root, dirs, files in os.walk(path):
    for file in files:
        if file == "cab_rides.csv":
            full_path = os.path.join(root, file)

print("Dataset path:", full_path)

# LOAD DATA
cab_data = pd.read_csv(full_path)

print("\nFirst few rows:")
print(cab_data.head())

print("\nDataset info:")
print(cab_data.info())


# DATA CLEANING
# Drop rows where price is missing
cab_data = cab_data.dropna(subset=["price"])

# Drop unnecessary columns
cab_data = cab_data.drop(columns=[
    "id", "product_id", "timestamp", "datetime"
], errors="ignore")

# FEATURE SELECTION
X = cab_data.drop("price", axis=1)
y = cab_data["price"]

# Convert categorical variables
X = pd.get_dummies(X, drop_first=True)

print("\nFeature shape after encoding:", X.shape)

# TRAIN-TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# SCALING
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# TRAIN MODEL (LINEAR REGRESSION)
model = LinearRegression()
model.fit(X_train, y_train)

# PREDICTIONS
y_pred = model.predict(X_test)


# EVALUATION METRICS
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n===== MODEL PERFORMANCE =====")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

# CONFIDENCE INTERVAL (BOOTSTRAP)
def bootstrap_ci(y_true, y_pred, n_bootstrap=1000):
    errors = []
    n = len(y_true)

    y_true = y_true.reset_index(drop=True)

    for _ in range(n_bootstrap):
        indices = np.random.choice(range(n), n, replace=True)
        error = np.mean(np.abs(y_true.iloc[indices] - y_pred[indices]))
        errors.append(error)

    return np.percentile(errors, [2.5, 97.5])

ci = bootstrap_ci(y_test, y_pred)

print("\n95% Confidence Interval for MAE:", ci)

100%|██████████| 73.5M/73.5M [00:01<00:00, 52.4MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/ravi72munde/uber-lyft-cab-prices/versions/4/cab_rides.csv

First few rows:
   distance cab_type     time_stamp    destination            source  price  \
0      0.44     Lyft  1544952607890  North Station  Haymarket Square    5.0   
1      0.44     Lyft  1543284023677  North Station  Haymarket Square   11.0   
2      0.44     Lyft  1543366822198  North Station  Haymarket Square    7.0   
3      0.44     Lyft  1543553582749  North Station  Haymarket Square   26.0   
4      0.44     Lyft  1543463360223  North Station  Haymarket Square    9.0   

   surge_multiplier                                    id    product_id  \
0               1.0  424553bb-7174-41ea-aeb4-fe06d4f4b9d7     lyft_line   
1               1.0  4bd23055-6827-41c6-b23b-3c491f24e74d  lyft_premier   
2               1.0  981a3613-77af-4620-a42a-0c0866077d1e          lyft   
3               1.0  c2d88af2-d278-4bfd-a8d0-29ca77cc5512   lyft_luxsuv   
4               1.0  e0126e1f